# Stable Diffusion from scratch - demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/qpp112/stable-diffusion-from-scratch/blob/main/notebooks/demo.ipynb)

Pure-PyTorch implementation of SD 1.5 and SDXL (UNet, VAE, CLIP text encoders, BPE tokenizer) with
hand-written DDPM / DDIM / DPM-Solver samplers. Only the *weights* come from the official checkpoints.

**Runtime:** GPU (Colab: *Runtime -> Change runtime type -> T4 / L4 / A100*). SDXL downloads ~7 GB on first use.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone -q https://github.com/qpp112/stable-diffusion-from-scratch.git
%cd stable-diffusion-from-scratch
!pip install -q -e . pytest regex

## 1. Unit tests (CPU, ~1 min)
Sampler order-of-accuracy checks, exact parameter counts of the official models, tiny end-to-end pipelines.

In [ ]:
!python -m pytest -q tests

## 2. Load SDXL
Pick a memory preset from the GPU size (see `docs/memory.md`).

In [ ]:
import torch
from sdscratch.pipeline import PRESETS, StableDiffusion

vram_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
preset = "fast" if vram_gb >= 15 else "balanced" if vram_gb >= 8 else "low"
print(f"{vram_gb:.0f} GB GPU -> preset '{preset}'")
pipe = StableDiffusion.from_pretrained("sdxl", memory=PRESETS[preset])

In [ ]:
res = pipe("a red fox in a snowy birch forest, golden hour, 85mm photograph",
           height=1024, width=1024, steps=25, sampler="dpm++2m", seed=0)
print(res.report.summary())
res.images[0]

## 3. Samplers side by side
Same prompt, seed and initial noise; only the solver changes.

In [ ]:
import matplotlib.pyplot as plt

samplers, nfes = ["ddpm", "ddim", "dpm++2m", "dpm++3m"], [8, 15, 25]
fig, axes = plt.subplots(len(samplers), len(nfes), figsize=(3 * len(nfes), 3 * len(samplers)))
for i, s in enumerate(samplers):
    for j, n in enumerate(nfes):
        img = pipe("a lighthouse on a cliff at dusk, oil painting", steps=n, sampler=s, seed=1,
                   height=768, width=768, progress=False).images[0]
        ax = axes[i, j]
        ax.imshow(img)
        ax.set_xticks([])
        ax.set_yticks([])
        if i == 0:
            ax.set_title(f"{n} NFE")
        if j == 0:
            ax.set_ylabel(s)
plt.tight_layout()

## 4. Memory options
Every option is a pure memory/speed trade-off - the image is unchanged.

In [ ]:
for opts in [dict(offload="none", vae_tiling=False), dict(offload="model", vae_tiling=True),
             dict(offload="model", vae_tiling=True, cfg_mode="sequential")]:
    pipe.set_memory_options(**opts)
    r = pipe("a red fox in a snowy birch forest, golden hour, 85mm photograph", steps=10, seed=0, progress=False)
    print(opts, "->", r.report.summary())
pipe.set_memory_options(PRESETS[preset])

## 5. Reproduce the README results
* `python scripts/benchmark_memory.py --model sdxl --size 1024` - VRAM / speed ladder (one subprocess per config)
* `python scripts/benchmark_memory.py --model sdxl --vram-limit-gb 8` - emulate an 8 GB consumer GPU
* `python scripts/compare_samplers.py --model sd15` - sampler grid + convergence on the real model
* `python scripts/parity_check.py --model sdxl` - numerical parity vs. `diffusers` (needs `pip install diffusers transformers accelerate`)
* `python scripts/toy_gmm.py` - exact-score sampler study (CPU)

In [ ]:
# free the pipeline first if GPU memory is tight:  del pipe; import gc; gc.collect(); torch.cuda.empty_cache()
!python scripts/benchmark_memory.py --model sdxl --size 1024 --steps 20 --skip-baseline